# EA3 — Proyecto final: procesamiento distribuido y cierre del caso

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | (15) |
| **Integrantes** | (Adolfo Miguel Jimenez) |
| **Caso de estudio** | (Wanderbricks) |
| **Fecha de entrega** | Domingo 4 de octubre |
| **🎥 Enlace al video** | *(pegar aquí — 8 a 10 minutos, mínimo 3 por integrante)* |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta
> a la propia. Un enlace inaccesible se califica como no entregado.

## Cómo usar esta plantilla

Las celdas en *cursiva* son instrucciones: **reemplácenlas por su contenido.**

| Sección | Qué va ahí | Puntos |
|---|---|---|
| 1 · Contexto | De dónde vienen, en una página | — |
| 2 · Capa oro | La tabla de consumo y el recorrido del dato | 20 |
| 3 · Medición inicial | El plan de ejecución y los tres tiempos | 15 |
| 4 · Optimización | La técnica, el plan nuevo y la comparación | 20 |
| 5 · Límites | Cuándo esa técnica sería contraproducente | 10 |
| 6 · Cierre del caso | La pregunta orientadora y las conclusiones | 10 |
| 7 · Reparto y uso de IA | Quién hizo qué | — |
| ⭐ Opcional | Tablero o Genie sobre la capa oro | +5 |

Los 25 puntos restantes son individuales y se evalúan en el video.

In [0]:
from pyspark.sql import functions as F
import time, statistics

# Catálogo y esquema configurados en el proyecto
CATALOGO = "bigdata_grupo15"
ESQUEMA  = "wanderbricks"

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")

print(f"Trabajando en {CATALOGO}.{ESQUEMA}\n")
display(spark.sql(f"SHOW TABLES IN {CATALOGO}.{ESQUEMA}"))

# 1 · Contexto

El presente proyecto se desarrolla sobre el caso de **Wanderbricks**, un marketplace de alquiler de propiedades vacacionales donde la actividad del negocio sigue el flujo **Usuario (`users`) → Reserva (`bookings`) → Pago (`payments`)**. En la **EA1** construimos las bases de nuestra arquitectura Medallion sobre Delta Lake: ingresamos los datos crudos hacia la capa **Bronce** (`users_bronce`, `bookings_bronce`, `payments_bronce`) añadiendo metadatos de auditoría, y aplicamos reglas de calidad hacia la capa **Plata** (`users_plata`, `bookings_plata`, `payments_plata`), donde depuramos la tabla de pagos pasando de **49.638 registros originales a 45.912 registros válidos** (descartando el **7,51 %** de datos inconsistentes). Posteriormente, en la **EA2** gobernamos este entorno dentro de Unity Catalog bajo el catálogo `bigdata_grupo15` y el esquema `wanderbricks`, aplicando controles de acceso mediante sentencias `GRANT`, verificando la trazabilidad del dato y automatizando la validación secuencial entre capas mediante el Job programado `Automatizacion_Wanderbricks` (`ID: 616036990504234`).

En esta última etapa (**EA3**) cerramos el pipeline construyendo la **capa Oro**, cuyo objetivo es responder directamente la pregunta de negocio planteada desde el inicio del proyecto: **¿Cuál es el método de pago predominante entre los usuarios que completan reservas en la plataforma y qué volumen de ingresos representa cada medio de pago?** Para lograrlo, integraremos la información depurada de usuarios, reservas y pagos en una tabla agregada final lista para el consumo directivo, con nombres de columnas claros, métricas financieras calculadas y ordenada de mayor a menor relevancia comercial.

# 2 · Capa oro · 20 puntos

## 2.1 · Qué pregunta responde

* **Pregunta de negocio:** ¿Cuál es el método de pago predominante entre los usuarios que realizan reservas en Wanderbricks, cuántos clientes únicos y transacciones concentra cada medio de pago, y qué volumen de ingresos (monto total, ticket promedio y porcentaje de participación) representa para la empresa?
* **Quién la usaría:** Los equipos de **Finanzas, Operaciones y Dirección Comercial** de Wanderbricks. Esta tabla les permite identificar de un vistazo qué pasarelas de pago concentran la mayor facturación para negociar mejores tarifas de comisión bancaria, priorizar el soporte técnico en los medios más utilizados y decidir qué métodos impulsar mediante campañas comerciales.
* **Por qué cumple con el estándar de Capa Oro:** Quien abre la tabla no se encuentra con llaves técnicas (`user_id`, `booking_id`, `payment_id`) ni etiquetas crudas del sistema (`credit_card`, `bank_transfer`), sino con los nombres comerciales claros de cada medio de pago, las métricas financieras ya consolidadas y las filas ordenadas de arriba hacia abajo empezando por el método líder del negocio.

In [0]:
# --- Construcción de la capa oro
df_users = spark.table("users_plata")
df_bookings = spark.table("bookings_plata")
df_payments = spark.table("payments_plata")

# 1. Cruzar Pagos -> Reservas -> Usuarios
df_cruce = (
    df_payments.alias("p")
    .join(df_bookings.alias("b"), on="booking_id", how="inner")
    .join(df_users.alias("u"), on="user_id", how="inner")
)

# 2. Nombres claros de negocio para los métodos de pago
df_nombres = df_cruce.withColumn(
    "metodo_de_pago",
    F.when(F.col("p.payment_method") == "credit_card", "Tarjeta de Crédito")
     .when(F.col("p.payment_method") == "paypal", "PayPal")
     .when(F.col("p.payment_method") == "apple_pay", "Apple Pay")
     .when(F.col("p.payment_method") == "google_pay", "Google Pay")
     .when(F.col("p.payment_method") == "bank_transfer", "Transferencia Bancaria")
     .otherwise(F.col("p.payment_method"))
)

# 3. Calcular total de pagos cruzados y construir métricas agregadas
total_pagos_cruzados = df_nombres.count()

df_oro = (
    df_nombres.groupBy("metodo_de_pago")
    .agg(
        F.count("p.payment_id").alias("total_transacciones"),
        F.countDistinct("b.booking_id").alias("reservas_pagadas"),
        F.countDistinct("u.user_id").alias("usuarios_unicos"),
        F.round(F.sum("p.amount"), 2).alias("ingresos_totales_usd"),
        F.round(F.avg("p.amount"), 2).alias("ticket_promedio_usd")
    )
    .withColumn(
        "participacion_pct",
        F.round((F.col("total_transacciones") / F.lit(total_pagos_cruzados)) * 100, 2)
    )
    .orderBy(F.col("total_transacciones").desc(), F.col("ingresos_totales_usd").desc())
)

# 4. Guardar como tabla Delta en la capa Oro
df_oro.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("metodos_pago_oro")
print("Tabla 'metodos_pago_oro' guardada exitosamente.")

## 2.2 · La tabla resultante

In [0]:
# TODO: display() de su capa oro
display(spark.table(f"{CATALOGO}.{ESQUEMA}.metodos_pago_oro"))


## 2.3 · El recorrido del dato

*Cuántos registros hay en cada capa y a qué corresponde cada diferencia. Si algo se
descartó, digan por qué.*

In [0]:
# Conteo por capa — ajusten los nombres de sus tablas
tablas_recorrido = [
    ("bronce", "users_bronce"),
    ("plata",  "users_plata"),
    ("bronce", "bookings_bronce"),
    ("plata",  "bookings_plata"),
    ("bronce", "payments_bronce"),
    ("plata",  "payments_plata"),
    ("oro",    "metodos_pago_oro")
]

for capa, tabla in tablas_recorrido:
    try:
        n = spark.table(f"{CATALOGO}.{ESQUEMA}.{tabla}").count()
        print(f"{capa:<8}{tabla:<28}{n:>12,} registros")
    except Exception as e:
        print(f"{capa:<8}{tabla:<28} ⚠️ ajustar el nombre")

**Interpretación: ¿a qué corresponde cada diferencia entre capas?**

* **Paso de Bronce a Plata (`49,638` → `45,912` registros en pagos):** Existe una reducción de **3,726 registros** (equivalente al **7.51 %** del total crudo en `payments_bronce`). Esta diferencia corresponde a la aplicación de las reglas de calidad y limpieza en la capa Plata, donde se descartaron transacciones duplicadas, registros con identificadores nulos (`payment_id` o `booking_id`), montos inválidos (`amount <= 0`) y estados de pago fallidos o anulados que no representan ingresos reales para Wanderbricks.
* **Cruce relacional dentro de Plata (`45,912` → `45,907` pagos integrados):** Al realizar el cruce interno (`inner join`) entre `payments_plata`, `bookings_plata` y `users_plata`, quedan **45,907 transacciones** válidas (la suma de `9,259 + 9,259 + 9,156 + 9,132 + 9,101`). Los **5 registros de diferencia** corresponden a pagos huérfanos cuya reserva o usuario asociado fue filtrado durante la depuración de las otras tablas de Plata, garantizando así la integridad referencial del flujo *Usuario → Reserva → Pago*.
* **Paso de Plata a Oro (`45,907` transacciones cruzadas → `5` registros en `metodos_pago_oro`):** La reducción a **5 filas finales** no corresponde a un descarte de información, sino a una **agregación de negocio (`groupBy`)** por la dimensión `metodo_de_pago` (*Tarjeta de Crédito, PayPal, Apple Pay, Google Pay y Transferencia Bancaria*). Esas 5 filas consolidan el 100 % de las 45,907 transacciones válidas, resumiendo el volumen de operaciones, reservas únicas, clientes únicos, ingresos totales y ticket promedio listos para la toma de decisiones.

# 3 · Medición inicial · 15 puntos

## 3.1 · La operación que vamos a optimizar

La operación que vamos a medir y optimizar es **la consulta original que construye nuestra capa Oro (`metodos_pago_oro`)**: el cruce (`inner join`) entre las tres tablas de la capa Plata (`payments_plata`, `bookings_plata` y `users_plata`), seguido de la transformación de nombres y la agregación con conteos distintos (`countDistinct` sobre reservas y usuarios) para obtener las métricas por método de pago.

**¿Por qué es costosa esta operación en Spark?**
1. **Doble cruce de tablas (`join`):** Requiere unir `payments_plata` (45.912 registros) con `bookings_plata` por `booking_id`, y el resultado volverlo a unir con `users_plata` por `user_id`.
2. **Múltiples conteos distintos (`countDistinct`):** Calcular simultáneamente `countDistinct("booking_id")` y `countDistinct("user_id")` dentro de la agrupación obliga a Spark a realizar intercambios de datos por red (*Shuffle* / `Exchange`) adicionales para eliminar duplicados antes de consolidar los 5 métodos de pago.

In [0]:
# --- Su consulta ANTES de optimizar (el mismo cruce y agregación de nuestra capa Oro)
df_users = spark.table(f"{CATALOGO}.{ESQUEMA}.users_plata")
df_bookings = spark.table(f"{CATALOGO}.{ESQUEMA}.bookings_plata")
df_payments = spark.table(f"{CATALOGO}.{ESQUEMA}.payments_plata")

consulta_original = (
    df_payments.alias("p")
    .join(df_bookings.alias("b"), on="booking_id", how="inner")
    .join(df_users.alias("u"), on="user_id", how="inner")
    .withColumn(
        "metodo_de_pago",
        F.when(F.col("p.payment_method") == "credit_card", "Tarjeta de Crédito")
         .when(F.col("p.payment_method") == "paypal", "PayPal")
         .when(F.col("p.payment_method") == "apple_pay", "Apple Pay")
         .when(F.col("p.payment_method") == "google_pay", "Google Pay")
         .when(F.col("p.payment_method") == "bank_transfer", "Transferencia Bancaria")
         .otherwise(F.col("p.payment_method"))
    )
    .groupBy("metodo_de_pago")
    .agg(
        F.count("p.payment_id").alias("total_transacciones"),
        F.countDistinct("b.booking_id").alias("reservas_pagadas"),
        F.countDistinct("u.user_id").alias("usuarios_unicos"),
        F.round(F.sum("p.amount"), 2).alias("ingresos_totales_usd"),
        F.round(F.avg("p.amount"), 2).alias("ticket_promedio_usd")
    )
    .orderBy(F.col("total_transacciones").desc(), F.col("ingresos_totales_usd").desc())
)

print("consulta_original definida con el código de la capa Oro.")

In [0]:
# --- Su consulta ANTES de optimizar
df_users = spark.table(f"{CATALOGO}.{ESQUEMA}.users_plata")
df_bookings = spark.table(f"{CATALOGO}.{ESQUEMA}.bookings_plata")
df_payments = spark.table(f"{CATALOGO}.{ESQUEMA}.payments_plata")

consulta_original = (
    df_payments.alias("p")
    .join(df_bookings.alias("b"), on="booking_id", how="inner")
    .join(df_users.alias("u"), on="user_id", how="inner")
    .withColumn(
        "metodo_de_pago",
        F.when(F.col("p.payment_method") == "credit_card", "Tarjeta de Crédito")
         .when(F.col("p.payment_method") == "paypal", "PayPal")
         .when(F.col("p.payment_method") == "apple_pay", "Apple Pay")
         .when(F.col("p.payment_method") == "google_pay", "Google Pay")
         .when(F.col("p.payment_method") == "bank_transfer", "Transferencia Bancaria")
         .otherwise(F.col("p.payment_method"))
    )
    .groupBy("metodo_de_pago")
    .agg(
        F.count("p.payment_id").alias("total_transacciones"),
        F.countDistinct("b.booking_id").alias("reservas_pagadas"),
        F.countDistinct("u.user_id").alias("usuarios_unicos"),
        F.round(F.sum("p.amount"), 2).alias("ingresos_totales_usd"),
        F.round(F.avg("p.amount"), 2).alias("ticket_promedio_usd")
    )
    .orderBy(F.col("total_transacciones").desc(), F.col("ingresos_totales_usd").desc())
)

print("consulta_original lista.")


## 3.2 · El plan de ejecución inicial

*Ejecuten `.explain()` y dejen la salida visible. Después señalen dónde está el costo.*

In [0]:
# TODO: consulta_original.explain()

consulta_original.explain()

**Qué vemos en el plan:**

*¿Qué estrategia eligió Spark? ¿Dónde aparece un `Exchange`? ¿Qué se está moviendo entre
máquinas y por qué eso es caro?*



Al leer el plan físico de abajo hacia arriba (desde la lectura de las tablas hasta el resultado final), identificamos qué estrategia eligió Spark (con el motor vectorizado Photon) y dónde está realmente el costo:

1. **Estrategia en los cruces (`PhotonBroadcastHashJoin`):** Como las tablas tienen estadísticas completas en Unity Catalog (`full = bookings_plata, payments_plata, users_plata`), Spark eligió automáticamente un *Broadcast Hash Join* para unir `bookings_plata` (`[id=#10912]`) y `users_plata` (`[id=#10925]`), enviándolas como una sola partición (`SinglePartition` / `EXECUTOR_BROADCAST`) y leyendo únicamente las columnas necesarias.
2. **Dónde aparecen los `Exchange` y qué se mueve entre máquinas:** El verdadero cuello de botella no está en el `join`, sino justo arriba de él, provocado por pedir **dos `countDistinct` sobre columnas distintas (`booking_id` y `user_id`)** en la misma agregación:
   * **`!PhotonExpand` (multiplicación de datos x3):** Como Spark no puede deduplicar dos columnas distintas en una sola pasada, inserta el operador `!PhotonExpand`, que **triplica cada fila en memoria** creando un identificador de grupo interno (`gid = 0`, `gid = 1`, `gid = 2`). Las 45.907 filas del cruce pasan a ser **137.721 filas**.
   * **Shuffle pesado (`PhotonShuffleExchangeSink hashpartitioning(metodo_de_pago, u.user_id, b.booking_id, gid, 16)` `[id=#10936]`):** Mueve por red esas 137.721 filas repartidas por usuario, reserva, método de pago y `gid` entre 16 particiones para poder eliminar duplicados.
   * **Segundo Shuffle de agregación (`PhotonShuffleExchangeSink hashpartitioning(metodo_de_pago, 16)` `[id=#10944]`):** Vuelve a redistribuir los datos por la red agrupando ahora solo por `metodo_de_pago`.
   * **Tercer Shuffle de ordenamiento (`PhotonShuffleExchangeSink rangepartitioning` `[id=#10950]`):** Mueve las filas finales para aplicar el ordenamiento global (`PhotonSort`).

## 3.3 · Los tres tiempos

In [0]:
# TODO
# tiempo_antes = medir("Consulta original", lambda: consulta_original.collect())

tiempo_antes = medir("Consulta original", lambda: consulta_original.collect())


# 4 · Optimización · 20 puntos

## 4.1 · Qué técnica aplicamos y por qué

*Una sola técnica, bien elegida. Expliquen por qué esa y no otra, en función de lo que
vieron en el plan inicial.*

**Técnica aplicada:** Sustitución de múltiples conteos distintos exactos (`F.countDistinct`) por **estimación de cardinalidad con HyperLogLog (`F.approx_count_distinct`)**.

**Justificación a partir del plan inicial:**
En el plan físico inicial (`consulta_original.explain()`), observamos que Spark y Photon ya resolvieron de forma óptima los cruces entre tablas mediante **`PhotonBroadcastHashJoin`** (aprovechando las estadísticas del catálogo) y ya filtraron únicamente las columnas requeridas. Por ende, aplicar un `broadcast()` manual o podar columnas antes del `join` no produciría ningún cambio en el plan de ejecución.

El verdadero cuello de botella revelado por el plan está en la etapa de agregación: al solicitar simultáneamente dos conteos distintos exactos sobre columnas diferentes (`booking_id` y `user_id`), Spark se ve obligado a insertar el operador **`!PhotonExpand`**, que **triplica el número de filas en memoria** (pasando de 45.907 a 137.721 registros mediante tres identificadores `gid`), seguido de un **`PhotonShuffleExchangeSink` adicional (`[id=#10936]`)** que redistribuye todas esas filas expandidas por la red antes de poder agrupar por `metodo_de_pago`.

Al aplicar `F.approx_count_distinct`, Spark reemplaza esa expansión y deduplicación por red con *sketches* en memoria en una sola pasada, **eliminando por completo el operador `!PhotonExpand` y suprimiendo una etapa entera de *Shuffle* (`Exchange`)**.

In [0]:
# --- Su consulta DESPUÉS de optimizar
consulta_optimizada = None   # TODO

consulta_optimizada = (
    df_payments.alias("p")
    .join(df_bookings.alias("b"), on="booking_id", how="inner")
    .join(df_users.alias("u"), on="user_id", how="inner")
    .withColumn(
        "metodo_de_pago",
        F.when(F.col("p.payment_method") == "credit_card", "Tarjeta de Crédito")
         .when(F.col("p.payment_method") == "paypal", "PayPal")
         .when(F.col("p.payment_method") == "apple_pay", "Apple Pay")
         .when(F.col("p.payment_method") == "google_pay", "Google Pay")
         .when(F.col("p.payment_method") == "bank_transfer", "Transferencia Bancaria")
         .otherwise(F.col("p.payment_method"))
    )
    .groupBy("metodo_de_pago")
    .agg(
        F.count("p.payment_id").alias("total_transacciones"),
        F.approx_count_distinct("b.booking_id").alias("reservas_pagadas"),
        F.approx_count_distinct("u.user_id").alias("usuarios_unicos"),
        F.round(F.sum("p.amount"), 2).alias("ingresos_totales_usd"),
        F.round(F.avg("p.amount"), 2).alias("ticket_promedio_usd")
    )
    .orderBy(F.col("total_transacciones").desc(), F.col("ingresos_totales_usd").desc())
)

print("consulta_optimizada lista.")

## 4.2 · El plan de ejecución resultante

In [0]:
# TODO: consulta_optimizada.explain()
consulta_optimizada.explain()

## 4.3 · Qué cambió entre los dos planes

| | Antes (`consulta_original`) | Después (`consulta_optimizada`) |
|---|---|---|
| **Estrategia de cruce** | `PhotonBroadcastHashJoin`, seguido del operador **`!PhotonExpand`** que triplica las filas en memoria (de 45.907 a 137.721) fuera de Photon para separar los dos `countDistinct`. | `PhotonBroadcastHashJoin` directo hacia `PhotonGroupingAgg`. **Desaparece `!PhotonExpand`** y todo corre de forma nativa en Photon. |
| **Movimiento de datos** | **3 *Shuffles* (`Exchange`):** uno pesado (`[id=#10936]`) que mueve las 137.721 filas por usuario y reserva, otro por método de pago (`[id=#10944]`) y el final (`[id=#10950]`). | **2 *Shuffles* (`Exchange`):** desaparece el *Shuffle* por usuario y reserva; queda uno solo de agregación (`[id=#17749]`) que envía *buffers* compactos (`HyperLogLog`). |
| **Ordenamiento** | `PhotonSort` (`[id=#10950]`) ejecutado después de 4 operadores `PhotonGroupingAgg` en cascada. | El mismo `PhotonSort` (`[id=#17755]`), pero resuelto tras una sola agregación parcial y final (2 operadores `PhotonGroupingAgg`). |

**Por qué baja el costo:** Al usar `approx_count_distinct`, eliminamos la triplicación de filas en memoria (`!PhotonExpand`) y ahorramos una etapa completa de intercambio de datos por red (*Shuffle*) junto con dos agregaciones intermedias.

## 4.4 · Los tiempos, comparados

In [0]:
# TODO

tiempo_despues = medir("Consulta optimizada", lambda: consulta_optimizada.collect())

mejora = (tiempo_antes - tiempo_despues) / tiempo_antes * 100
print("═" * 48)
print(f"{'Antes':<24}{tiempo_antes:>8.2f}s")
print(f"{'Después':<24}{tiempo_despues:>8.2f}s")
print(f"{'Mejora':<24}{mejora:>8.1f}%")
print("═" * 48)

**Conclusión de la optimización:**

*Qué hacía antes, qué cambió en el plan, y el número. En ese orden.*

> 💡 **Si no mejoró nada, repórtenlo igual** y expliquen por qué: volumen insuficiente,
> Spark ya lo estaba resolviendo solo, u otra razón. Una medición honesta vale más que un
> número inflado.

%md
**Conclusión de la optimización:**

1. **Qué hacía antes:** Al pedir dos conteos distintos exactos (`countDistinct` sobre `booking_id` y `user_id`) en la misma agrupación, Spark insertaba el operador `!PhotonExpand` (fuera de Photon) para triplicar las filas en memoria (de 45.907 a 137.721 registros) y ejecutaba una etapa adicional de *Shuffle* (`hashpartitioning` por método de pago, usuario, reserva y `gid`) escoltada por dos agregaciones intermedias para eliminar duplicados.
2. **Qué cambió en el plan:** Al reemplazar `countDistinct` por `approx_count_distinct`, desapareció por completo el operador `!PhotonExpand`, toda la consulta pasó a ejecutarse de forma 100 % vectorizada en Photon y se eliminó esa etapa intermedia de *Shuffle* junto con dos nodos `PhotonGroupingAgg`, enviando por red únicamente *buffers* compactos de HyperLogLog agrupados por `metodo_de_pago`.
3. **El número y por qué la mejora en segundos es moderada:** La mediana pasó de **1.28 s a 1.22 s** (**4.3 % de mejora**, con un mínimo de **0.937 s** en la primera corrida frente a **1.191 s** antes). El cambio en reloj es pequeño porque el volumen de datos es reducido (~45.907 registros) y Spark ya había optimizado automáticamente los cruces con `PhotonBroadcastHashJoin`; en un clúster *Serverless*, la sobrecarga fija de coordinación (~1 s) domina el tiempo total, aunque el ahorro estructural en red y memoria se multiplicaría al escalar a millones de transacciones.

# 5 · Límites de la optimización · 10 puntos

Usar el conteo aproximado dejaría de servir o daría peores resultados en nuestro proyecto en tres casos:

* **Cuando los resultados están muy parejos:** En Wanderbricks los cinco métodos de pago tienen números casi idénticos. Entre Tarjeta de Crédito (8.928 usuarios) y PayPal (8.888 usuarios) solo hay 40 clientes de diferencia. Como el cálculo aproximado tiene un margen de error del 5 %, puede desviarse por más de 400 usuarios y alterar el orden real de qué método de pago se usa más.
* **Cuando se requiere exactitud contable:** Si el equipo de finanzas usa esta tabla para auditorías o para pagar comisiones exactas a los bancos por cada reserva, una aproximación no sirve porque descuadraría los cierres financieros.
* **Si solo contáramos un dato único o pidiéramos precisión máxima:** Esta mejora funcionó porque contábamos usuarios únicos y reservas únicas al mismo tiempo, lo que hacía que Spark triplicara las filas. Si solo necesitáramos contar uno de los dos, o si configuráramos la fórmula para tener un error casi de cero con tan pocos datos, el proceso tardaría igual o más que el conteo normal.

# 6 · Cierre del caso · 10 puntos

## 6.1 · La pregunta orientadora

Para integrar fuentes tan variadas como un portal web, redes sociales y WhatsApp Business, la empresa debería trabajar bajo el paradigma **Lakehouse** aplicando una **arquitectura Medallion dividida en tres capas (Bronce, Plata y Oro)**, tal como construimos en este proyecto para Wanderbricks. En la **capa Bronce** se recibirían los datos crudos de cada canal sin modificarlos, guardando tanto los registros del portal web (como hicimos en la EA1 con usuarios, reservas y pagos) como los mensajes e interacciones de Instagram, Facebook, TikTok y WhatsApp con su fecha de carga para asegurar la trazabilidad. Luego, en la **capa Plata**, se estandarizarían los formatos y se aplicarían reglas de limpieza para descartar datos repetidos, vacíos o con errores, garantizando que solo pase información confiable antes de cruzar los canales.

En cuanto a las herramientas, la solución ideal es procesar los datos de forma distribuida con **PySpark dentro de Databricks** y almacenarlos en formato **Delta Lake**, lo que permite manejar grandes volúmenes sin perder consistencia. Para que la plataforma sea segura y funcione sola en el día a día, todo debe administrarse con **Unity Catalog** para controlar los permisos de acceso de cada área y con **Jobs programados** para automatizar la actualización entre capas, igual que implementamos en la EA2. Finalmente, en la **capa Oro** se entregarían tablas ya resumidas y en lenguaje claro para que el equipo comercial tome decisiones directas —por ejemplo, qué red social genera más reservas reales y pagos completados—, verificando siempre el plan de ejecución de las consultas para evitar cruces o conteos lentos a medida que crezcan los datos, como aplicamos en esta EA3.

## 6.2 · Conclusiones del proyecto

* **Qué funcionó:** La limpieza por capas nos permitió filtrar desde el inicio el 7,51 % de los pagos con errores (pasando de 49.638 a 45.912 registros) y detectar 5 pagos sin reserva válida al cruzar las tablas en la capa Plata. Con esto logramos resumir la operación en una tabla Oro de 5 filas claras para el negocio, donde identificamos que Tarjeta de Crédito y PayPal empatan en el primer lugar con 9.259 pagos cada uno, pero Tarjeta de Crédito genera más ingresos ($5.132.271,63 USD). Además, al revisar el plan de ejecución y cambiar el conteo exacto por uno aproximado, logramos evitar que Spark triplicara las filas en memoria.
* **Qué no funcionó como esperábamos:** En la medición de tiempos la mejora fue mínima, bajando apenas de 1,28 s a 1,22 s (un 4,3 %). Esto ocurrió porque con 45.907 registros el motor de Databricks ya optimizaba solo los cruces de tablas y la mayor parte del segundo se va en el arranque normal del entorno Serverless. Tampoco resultó práctico usar el conteo aproximado para definir la tabla final, porque los cinco métodos de pago tienen cantidades tan parecidas que un margen de error del 5 % altera el orden real entre el primer y el último lugar.
* **Qué haríamos distinto si empezáramos de nuevo:** Primero, agregaríamos en la capa Plata una validación para cruzar las llaves de pagos con reservas desde la limpieza inicial, evitando que esos 5 registros huérfanos lleguen hasta el cálculo de la capa Oro. Segundo, no optimizaríamos a ciegas antes de ver el plan de ejecución: mantendríamos el conteo exacto para construir la tabla Oro oficial de finanzas (donde se requiere precisión total y el volumen actual tarda poco más de un segundo) y dejaríamos las funciones aproximadas únicamente para consultas exploratorias cuando la base de datos escale a millones de registros.

# 7 · Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| **Adolfo Jiménez** | Desarrollo completo de la evidencia: creación de la tabla `metodos_pago_oro`, validación entre capas, análisis de `.explain()`, optimización con `approx_count_distinct` y conclusiones. | Sustentación integral del notebook: resultados de la capa Oro, comparación de planes de ejecución, tiempos medidos y límites de la optimización. |

**Uso de asistentes de IA:** 
Se utilizó **Gemini** y el **asistente de IA de Databricks (Genie)** como apoyo para consultar la sintaxis de PySpark (`F.when` y `F.approx_count_distinct`), interpretar el operador `!PhotonExpand` del plan físico y dar formato a las tablas Markdown. Toda la ejecución del código, las mediciones de tiempo y el análisis de los datos de Wanderbricks fueron realizados y validados de forma individual en la plataforma.

# ⭐ Opcional · hasta 5 puntos adicionales

*Elijan **una** de las dos opciones. Es opcional: no hacerlo no resta nada.*

### Opción A · Un tablero sobre la capa oro

*Tres visualizaciones que respondan la pregunta de negocio de su capa oro. Peguen la captura
aquí.*

### Opción B · Un espacio de Genie sobre la capa oro

*Tres preguntas en lenguaje natural con sus respuestas, y **qué tuvieron que ajustar** para
que Genie entendiera: nombres de columnas, descripciones, instrucciones del espacio.*

*Eso último es lo más interesante del ejercicio: si Genie no entiende su tabla, es porque la
tabla no está bien diseñada.*

# ⭐ Opcional · hasta 5 puntos adicionales

### Opción B · Un espacio de Genie sobre la capa oro

**1. Tres preguntas en lenguaje natural y sus respuestas en Genie:**

* **Pregunta 1:** *¿Cuál es el método de pago que genera más ingresos en dólares y cuánto factura?*
  * **Respuesta:** Tarjeta de Crédito con $5.132.271,63 USD (20,17 % del total).

![pregunta 1.png](./pregunta 1.png "pregunta 1.png")!

* **Pregunta 2:** *¿Qué métodos de pago tienen la mayor cantidad de transacciones completadas?*
  * **Respuesta:** Tarjeta de Crédito y PayPal empatan en el primer lugar con 9.259 transacciones cada uno.

![pregunta 2.png](./pregunta 2.png "pregunta 2.png")!

* **Pregunta 3:** *¿Cuál es el método de pago con el ticket promedio más bajo y cuántos usuarios únicos tiene?*
  * **Respuesta:** Transferencia Bancaria tiene el ticket promedio más bajo ($552,65 USD) con 8.786 usuarios únicos.

![pregunta 3.png](./pregunta 3.png "pregunta 3.png")!

**2. Qué ajustamos para que Genie entendiera:**
* **Nombres claros desde el origen:** Como en la tabla `metodos_pago_oro` pusimos nombres de columnas en español (`metodo_de_pago`, `total_transacciones`, `ingresos_totales_usd`, `ticket_promedio_usd`) y tradujimos las categorías, Genie identificó los datos al primer intento.
* **Instrucciones del espacio:** Agregamos una nota indicando que la tabla ya viene resumida por método de pago en dólares (USD), evitando que Genie intentara volver a agrupar filas o sumar la columna de porcentajes.


---
# ✅ Antes de entregar

- [ ] El notebook corre completo de arriba abajo sin errores
- [ ] La capa oro existe y sus columnas se entienden sin explicación
- [ ] Las **tres mediciones** están visibles, antes y después, con la mediana
- [ ] Los **dos planes de ejecución** están en el notebook, con la salida visible
- [ ] Está escrito por qué mejoró y cuándo la técnica sería contraproducente
- [ ] La pregunta orientadora está respondida, apoyada en lo que construyeron
- [ ] No quedaron textos en cursiva de esta plantilla sin reemplazar
- [ ] La sección 7 coincide con lo que cada uno sustenta en el video
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] El `.ipynb` está confirmado en el repositorio, en `/ea3`
- [ ] El HTML con salidas visibles está subido a Canvas

### ⚠️ Sobre la cuota

Este proyecto pide ejecutar la misma consulta **ocho veces** —calentamiento y tres
mediciones, dos veces—. Eso gasta bastante más de lo habitual.
**No dejen las mediciones para la última noche.**

---

**Entrega: domingo 4 de octubre, 11:59 p. m.**